# Ejecutor de Colab

Notebook para que los integrantes **sin GPU** aporten cómputo desde Google Colab (issue 006). Hace, en orden:

1. Monta la carpeta compartida de Google Drive.
2. Clona el repositorio en un **commit fijo** (no se copia código en celdas).
3. Instala las dependencias fijadas en `requirements.txt` **sin reinstalar PyTorch** (se usa el de Colab).
4. Verifica la GPU.
5. Verifica la huella (MD5 y tamaño) de los tokenizadores y los datos tokenizados de Drive, y se detiene si no coinciden.
6. Lanza un fragmento de búsqueda con `buscar.py` (o un entrenamiento: pendiente del issue 009).
7. Guarda los resultados **directamente en Drive**: la caché JSON se reescribe tras cada evaluación, así que si Colab corta la sesión basta con volver a ejecutar todo el notebook y continúa sin repetir evaluaciones.

**Uso en Colab:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*, editar solo la celda de configuración y ejecutar todo (*Entorno de ejecución → Ejecutar todas*).

**Modo local:** fuera de Colab, el notebook simula Drive en una carpeta temporal, clona el repositorio local en su `HEAD` y crea artefactos simulados. Sirve para probar el flujo con `APTITUD = "falsa"` sin GPU.

> **Azure:** descartado. Las cuentas *Azure for Students* del equipo tienen cuota 0 de GPU utilizable (ver `issues/done/002-verificar-cuota-gpu-azure.md`), así que no hay versión para VM de Azure.

## 0. Configuración

Lo único que se edita. El resto de celdas no necesita cambios.

In [ ]:
import sys

# Modo: "colab" en Google Colab; "local" simula Drive en una carpeta temporal (pruebas sin GPU).
MODO = "colab" if "google.colab" in sys.modules else "local"

# Carpeta compartida de Drive (tal como aparece tras montarla). TODO: ruta real cuando exista.
CARPETA_DRIVE = "/content/drive/MyDrive/TODO-carpeta-compartida"

# Repositorio y commit fijado (todas las cuentas deben usar el mismo).
URL_REPO = "https://github.com/FROY33/Sistema-de-traduccion-automatica"
COMMIT = "024752a6585bf19c614ca04828e7b8fc351bc02e"

# Qué ejecutar.
TAREA = "busqueda"          # "busqueda" o "entrenamiento" (TODO issue 009)
METODO = "aleatoria"        # "aleatoria", "mmas" o "defecto"
PAR = "en-la"               # "en-la" o "en-el"
SEMILLA = 0
FRAGMENTO, FRAGMENTOS = 1, 3  # solo para METODO = "aleatoria": cada cuenta, su fragmento
APTITUD = "falsa"           # "falsa" (sin GPU) o "real" (issue 003, aún no implementada)
MODELO_A_ENTRENAR = None    # TODO issue 009: p. ej. "linea_base", "mejor_aleatorio", "mejor_mmas"

INSTALAR_DEPENDENCIAS = True

# Huella esperada de los artefactos: copiar los valores anotados en
# issues/001-regenerar-artefactos-y-entorno.md. Con None la verificación se detiene
# (salvo con APTITUD = "falsa", que no usa artefactos).
HUELLA_ESPERADA = {
    "en-la": {"md5_tokenizador": None, "md5_datos": None, "bytes_datos": None},
    "en-el": {"md5_tokenizador": None, "md5_datos": None, "bytes_datos": None},
}

## 1. Montar Drive

En modo local se usa una carpeta temporal en lugar de Drive.

In [ ]:
import subprocess
import tempfile
from pathlib import Path

EN_COLAB = MODO == "colab"

if EN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    CARPETA_REPO = Path("/content/Sistema-de-traduccion-automatica")
    if not Path(CARPETA_DRIVE).is_dir():
        raise FileNotFoundError(
            f"No existe {CARPETA_DRIVE}. Revisa CARPETA_DRIVE (¿añadiste el acceso directo a la carpeta compartida en tu Drive?)."
        )
else:
    BASE_LOCAL = Path(tempfile.gettempdir()) / "ejecutor_colab_local"
    CARPETA_DRIVE = str(BASE_LOCAL / "drive")
    CARPETA_REPO = BASE_LOCAL / "repo"
    # Se clona el repositorio local (la carpeta que contiene este notebook) en su HEAD.
    URL_REPO = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, check=True
    ).stdout.strip()
    COMMIT = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=URL_REPO, capture_output=True, text=True, check=True
    ).stdout.strip()
    Path(CARPETA_DRIVE).mkdir(parents=True, exist_ok=True)

print(f"Modo: {MODO}")
print(f"Drive: {CARPETA_DRIVE}")
print(f"Repositorio: {URL_REPO} @ {COMMIT}")

## 2. Clonar el repositorio en el commit fijo

Si ya estaba clonado (sesión relanzada), solo se actualiza y se vuelve al commit fijado. Después se comprueba que `HEAD` es ese commit y que no hay cambios sin confirmar.

In [ ]:
import importlib


def git(*argumentos, carpeta=None):
    subprocess.run(["git", *argumentos], cwd=carpeta, check=True)


if not (CARPETA_REPO / ".git").exists():
    git("clone", "--quiet", URL_REPO, str(CARPETA_REPO))
else:
    git("fetch", "--quiet", "origin", carpeta=CARPETA_REPO)
git("checkout", "--quiet", "--detach", COMMIT, carpeta=CARPETA_REPO)

if str(CARPETA_REPO) not in sys.path:
    sys.path.insert(0, str(CARPETA_REPO))
import traduccion.ejecutor as ejecutor

ejecutor = importlib.reload(ejecutor)  # por si se cambió de commit en la misma sesión

COMMIT_COMPLETO = ejecutor.comprobar_commit(CARPETA_REPO, COMMIT)
print(f"Repositorio en {COMMIT_COMPLETO} (sin cambios sin confirmar)")

## 3. Instalar dependencias

Se instalan las versiones fijadas de `requirements.txt` **excepto** `torch` (Colab trae el suyo, compilado para su CUDA) y el kernel de Jupyter (`ipykernel`, `jupyter`). La búsqueda corre en un proceso aparte, así que no hace falta reiniciar el entorno aunque cambien versiones ya cargadas. Si la versión de PyTorch difiere de la fijada, se avisa: la versión real queda registrada en la huella de cada resultado.

In [ ]:
TEXTO_REQUIREMENTS = (CARPETA_REPO / "requirements.txt").read_text(encoding="utf-8")
DEPENDENCIAS = ejecutor.dependencias_para_colab(TEXTO_REQUIREMENTS)

if EN_COLAB and INSTALAR_DEPENDENCIAS:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *DEPENDENCIAS], check=True)
    print("Dependencias instaladas:", ", ".join(DEPENDENCIAS))
else:
    print("No se instala nada (modo local o INSTALAR_DEPENDENCIAS = False). En Colab se instalaría:")
    print("  pip install " + " ".join(DEPENDENCIAS))

VERSION_TORCH = ejecutor.version_instalada("torch")
AVISO_TORCH = ejecutor.aviso_version_torch(TEXTO_REQUIREMENTS, VERSION_TORCH)
print(f"PyTorch: {VERSION_TORCH}")
if AVISO_TORCH:
    print(f"AVISO: {AVISO_TORCH}")

## 4. Verificar la GPU

La aptitud real y el entrenamiento exigen GPU; con la aptitud falsa solo se informa.

In [ ]:
GPU = ejecutor.nombre_gpu()
REQUIERE_GPU = APTITUD == "real" or TAREA == "entrenamiento"
print(f"GPU: {GPU or 'ninguna'}")

if GPU is None and REQUIERE_GPU:
    raise RuntimeError(
        "No hay GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno → GPU T4, y volver a ejecutar todo."
    )
if GPU is not None and EN_COLAB and "T4" not in GPU:
    print(f"AVISO: la GPU no es una T4 ({GPU}); queda registrada en el registro de la ejecución.")

## 5. Verificar la huella de los artefactos

Calcula el MD5 del tokenizador del par, el MD5 de la carpeta de datos tokenizados (rutas relativas y MD5 de cada archivo) y su tamaño en bytes, y **se detiene** si no coinciden con `HUELLA_ESPERADA` (los valores del issue 001). Estructura esperada en Drive: `artefactos/tok_la_8000.model`, `artefactos/tok_el_16000.model`, `artefactos/datos/en_la/`, `artefactos/datos/en_el/`.

En modo local se crean artefactos simulados y se toman sus propios valores como esperados.

In [ ]:
RUTAS = ejecutor.RutasEjecutor(Path(CARPETA_DRIVE))
RUTAS.crear()

if not EN_COLAB:
    ejecutor.crear_artefactos_simulados(RUTAS, PAR)
    HUELLA_ESPERADA[PAR] = ejecutor.huella_artefactos(RUTAS, PAR)

USA_ARTEFACTOS = APTITUD == "real" or TAREA == "entrenamiento"
anotada = all(HUELLA_ESPERADA[PAR].get(c) is not None for c in ejecutor.CAMPOS_ARTEFACTOS)

if USA_ARTEFACTOS or anotada:
    HUELLA_ARTEFACTOS = ejecutor.verificar_artefactos(RUTAS, PAR, HUELLA_ESPERADA[PAR])
    print("Artefactos verificados:")
    for campo, valor in HUELLA_ARTEFACTOS.items():
        print(f"  {campo}: {valor}")
else:
    HUELLA_ARTEFACTOS = None
    print("Aptitud falsa y huella sin anotar: no se usan artefactos, se omite la verificación.")

## 6. Lanzar el trabajo

**Búsqueda:** ejecuta `buscar.py` desde el repositorio clonado con `--salida` en Drive. Si la sesión se corta, volver a ejecutar todo el notebook con la misma configuración continúa desde la caché sin repetir evaluaciones.

**Entrenamiento:** pendiente del issue 009.

In [ ]:
import time

inicio = time.time()
if TAREA == "busqueda":
    COMANDO = ejecutor.comando_busqueda(
        CARPETA_REPO, RUTAS, METODO, SEMILLA, PAR, APTITUD, FRAGMENTO, FRAGMENTOS
    )
    print("$ " + " ".join(COMANDO) + "\n")
    CODIGO_SALIDA = ejecutor.ejecutar_con_salida(COMANDO, CARPETA_REPO)
elif TAREA == "entrenamiento":
    # TODO issue 009: lanzar el entrenamiento completo de MODELO_A_ENTRENAR con el script
    # de entrenamiento final (aún no existe) y guardar puntos de control en RUTAS.resultados
    # para poder reanudar tras un corte de Colab.
    raise NotImplementedError("TODO issue 009: el entrenamiento completo aún no está implementado.")
else:
    raise ValueError(f"TAREA desconocida: {TAREA!r}")

REGISTRO = ejecutor.registrar_ejecucion(
    RUTAS,
    {
        "tarea": TAREA,
        "metodo": METODO,
        "par": PAR,
        "semilla": SEMILLA,
        "fragmento": FRAGMENTO,
        "fragmentos": FRAGMENTOS,
        "aptitud": APTITUD,
        "modo": MODO,
        "commit": COMMIT_COMPLETO,
        "gpu": GPU,
        "torch": VERSION_TORCH,
        "huella_artefactos": HUELLA_ARTEFACTOS,
        "comando": COMANDO,
        "codigo_salida": CODIGO_SALIDA,
        "duracion_s": round(time.time() - inicio, 1),
    },
)
print(f"\nRegistro de la ejecución: {REGISTRO}")
if CODIGO_SALIDA != 0:
    raise RuntimeError(f"buscar.py terminó con código {CODIGO_SALIDA}")

## 7. Resultados en Drive

Resume el JSON del fragmento y muestra la huella de un resultado (debe indicar el commit fijado). Después, quien coordine reúne los JSON de las 3 cuentas con `fusionar.py` (ver README, sección 11).

In [ ]:
import json

from traduccion import busqueda

RUTA_RESULTADOS = busqueda.ruta_resultados(RUTAS.resultados, PAR, APTITUD, METODO, SEMILLA, FRAGMENTO, FRAGMENTOS)
resultados = json.loads(RUTA_RESULTADOS.read_text(encoding="utf-8"))["resultados"]
mejor = max(resultados.values(), key=lambda r: r["aptitud"])

print(f"Archivo: {RUTA_RESULTADOS}")
print(f"Evaluaciones guardadas: {len(resultados)}")
print(f"Mejor aptitud: {mejor['aptitud']:.4f}")
print("Huella de un resultado:")
print(json.dumps(mejor["huella"], ensure_ascii=False, indent=2))
assert mejor["huella"]["commit"] == COMMIT_COMPLETO, "La huella no indica el commit fijado"